In [ ]:
import numpy as np
import pandas as pd
import ruptures as rpt
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

In [ ]:
file='DataExport132110.csv'
path = 'processed_data/'+file

In [ ]:
df = pd.read_csv(path, parse_dates=['timestamp'], index_col='timestamp')
df.index = pd.to_datetime(df.index)
# Make sure the index is sorted
df = df.sort_index()
df.drop(columns=['Unnamed: 0'])
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]  

In [ ]:
df.head()

In [ ]:
# given df using timestampl give that first date and final date 
first_date = df.index[0]
last_date = df.index[-1]

# If you want just the date portion without time:
print(f"First date: {first_date.date()}")
print(f"Last date: {last_date.date()}")

### Hyperparameters

Explanation of each hyperparameter in the change detection algorithm:

1. `base_penalty` (default=5.0):
   - Controls how sensitive the algorithm is to detecting changes
   - Higher values result in fewer change points (more conservative)
   - Lower values result in more change points (more sensitive)
   - Works with adaptive_penalty function to scale based on the signal's standard deviation

2. `model` (default='l2'):
   - Defines the cost function used to detect changes
   - 'l2': Uses mean squared error (good for detecting changes in mean)
   - Other options include: 'l1' (mean absolute error), 'rbf' (radial basis function)
   - 'l2' is generally most suitable for detecting level shifts in time series

3. `min_duration` (default=30):
   - Minimum number of data points required between change points
   - With 15-minute data, 30 points = 7.5 hours
   - Helps filter out short-term fluctuations
   - Higher values ensure detected changes are more persistent
   - Calculate points for 5 days: 1 day = 24 hours * 4 (15-min intervals) = 96 points per day

4. `min_pct_change` (default=5.0):
   - Minimum percentage change required to register as a significant change
   - Example: 5.0 means a 5% change in the mean value
   - Higher values will only detect larger changes
   - Lower values will detect more subtle changes

These two parameters serve different purposes in the change detection algorithm:

`base_penalty` (default=3.0):
- Controls the sensitivity of the change point detection algorithm itself
- Affects how many change points are initially detected
- Lower values (e.g., 1.0) will detect more potential change points
- Higher values (e.g., 10.0) will detect fewer change points
- Works with the signal's standard deviation in adaptive_penalty function

`min_pct_change` (default=10.0):
- Filters the detected changes based on the percentage difference between segments
- Only keeps changes where the difference between segment means exceeds this percentage
- Example: 10.0 means only keep changes where there's at least a 10% increase
- Applied after change points are detected
- Higher values (e.g., 20.0) will only show larger changes
- Lower values (e.g., 5.0) will show more subtle changes

In practice:
- `base_penalty` affects the initial segmentation
- `min_pct_change` filters those segments based on the size of the change

In [ ]:
# Define hyperparameters
days=7
HYPERPARAMETERS = {
    'base_penalty': 3.0,
    'model': 'l2',
    'min_duration': 96*days, # 5 days 480=96*5
    'min_pct_change': 10.0
}


### Functions

In [ ]:
def adaptive_penalty(signal, base=1.0):
    return base * np.nanstd(signal)

def detect_persistent_changes(name, series, params=HYPERPARAMETERS):
    series = series.sort_index().asfreq('15T')
    signal = series.values
    idx = series.index

    if np.isnan(signal).any():
        signal = pd.Series(signal).interpolate().fillna(method="bfill").fillna(method="ffill").values

    penalty = adaptive_penalty(signal, base=params['base_penalty'])
    algo = rpt.Binseg(model=params['model']).fit(signal)
    change_idxs = algo.predict(pen=penalty)

    if change_idxs[-1] == len(signal):
        change_idxs = change_idxs[:-1]

    segments = list(zip([0] + change_idxs[:-1], change_idxs))
    persistent_changes = []

    prev_mean = None
    for (start, end) in segments:
        duration = end - start
        if duration < params['min_duration']:
            continue

        segment_mean = signal[start:end].mean()
        if prev_mean is not None:
            pct_change = 100 * (segment_mean - prev_mean) / prev_mean
            if abs(pct_change) >= params['min_pct_change']:
                change_time = idx[start]
                persistent_changes.append({
                    "change_time": change_time,
                    "step_pct": round(pct_change, 2),
                    "direction": "up" if pct_change > 0 else "down",
                    "sustain_duration": f"{duration * 15} minutes"
                })

        prev_mean = segment_mean

    return {
        "meter": name,
        "has_change": len(persistent_changes) > 0,
        "changes": persistent_changes
    }

In [ ]:
def detect_upward_changes(name, series, params=HYPERPARAMETERS):
    series = series.sort_index().asfreq('15T')
    signal = series.values
    idx = series.index

    if np.isnan(signal).any():
        signal = pd.Series(signal).interpolate().fillna(method="bfill").fillna(method="ffill").values

    penalty = adaptive_penalty(signal, base=params['base_penalty'])
    algo = rpt.Binseg(model=params['model']).fit(signal)
    change_idxs = algo.predict(pen=penalty)

    if change_idxs[-1] == len(signal):
        change_idxs = change_idxs[:-1]

    segments = list(zip([0] + change_idxs[:-1], change_idxs))
    upward_changes = []

    prev_segment_avg = None
    for (start, end) in segments:
        duration = end - start
        if duration < params['min_duration']:
            continue

        # Calculate average over the minimum duration period
        min_duration_end = min(start + params['min_duration'], end)
        segment_avg = signal[start:min_duration_end].mean()
        
        if prev_segment_avg is not None:
            pct_change = 100 * (segment_avg - prev_segment_avg) / prev_segment_avg
            # Only record positive changes
            if pct_change >= params['min_pct_change']:
                change_time = idx[start]
                upward_changes.append({
                    "change_time": change_time,
                    "step_pct": round(pct_change, 2),
                    "sustain_duration": f"{duration * 15} minutes"
                })

        prev_segment_avg = segment_avg

    return {
        "meter": name,
        "has_change": len(upward_changes) > 0,
        "changes": upward_changes
    }


In [ ]:
def plot_meter_with_changes(series, changes, title=''):
    plt.figure(figsize=(12, 4))
    plt.plot(series.index, series.values, label='Meter Reading', color='black')

    for change in changes:
        plt.axvline(change['change_time'], color='red', linestyle='--', label='Detected Change')
        plt.text(change['change_time'], max(series),
                 f"{change['step_pct']}%",
                 rotation=90, verticalalignment='bottom', fontsize=8)

    plt.title(title)
    plt.ylabel('Value')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

### Calculation

In [ ]:
results = Parallel(n_jobs=-1)(
    delayed(detect_upward_changes)(name, df[name])
    for name in df.columns
)


In [ ]:
df_results = pd.DataFrame(results)
df_results = df_results[df_results['has_change'] == True]
print(df_results.shape)
df_results.head()

### Visualisation

In [ ]:
# Loop through all results where there are changes
for result in results:
    if result['has_change']:  # or result['has_upward_change'] if using detect_upward_changes
        meter = result['meter']
        print(f"Plotting {meter}")
        
        # Create a new figure for each plot
        plt.figure(figsize=(15, 5))
        plot_meter_with_changes(df[meter], result['changes'], title=meter)
        plt.show()
        
        # Optional: add a small pause between plots
        plt.pause(0.5)

### Specific meters

In [ ]:
# meter = "meter1"

# Find the result for the specific meter
result = next(r for r in results if r['meter'] == meter)

# Plot with the changes
plot_meter_with_changes(df[meter], result['changes'], title=result['meter'])
